## How to run this notebook

There are two ways. Both are safe; choose either.

**Way 1: run your own copy.** No warning, and your changes are kept.

- At the top of this page, choose **Copy to Drive** (or **File → Save a copy in Drive**).
- Your copy opens in a new tab. It is saved in your Google Drive, in the folder **Colab Notebooks**.
- In your copy, choose **Runtime → Run all**.

**Way 2: run this page directly.** Quicker, but your changes are not kept.

- Choose **Runtime → Run all**.
- Colab shows this warning:

  > Warning: This notebook was not authored by Google. This notebook is being loaded from GitHub. It may request access to your data stored with Google, or read data and credentials from other sessions. Please review the source code before executing this notebook.

- Choose **Run anyway**.

**Why it is safe.** Colab shows this warning for every notebook that does not come from Google, before it has looked at the code. This notebook does not ask for your Google account, your Drive or any other data of yours. It only saves the table it analyses, and any figures it draws, in this session, which ends when you close the page. You can read every line of the code below before you run it.

# ZipEats pitch claims: mean intervals from a summary

**Confidence Interval Estimation for the Mean** · a worked example from the StatNex teaching guide

**The story.** ZipEats is a budget food-delivery app. A manager emailed only a summary of 50 sampled orders: the sample size, mean and standard deviation of delivery time, distance, order total and tip. You can build the confidence intervals from this summary.

**The question.** Can we build the confidence intervals from a three-line summary, without the raw orders?

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_mean_ci__summary.csv` (3 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Variable,Delivery time (min),Distance (km),Order total ($),Tip ($)
Sample size,50.0,50.0,50.0,50.0
Mean,24.46,1.92,15.264,1.2592
Standard deviation,6.4908,0.6478,3.7695,0.8845
""")

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)

## The data

In [ ]:
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["Delivery time (min)", "Distance (km)", "Order total ($)", "Tip ($)"]
confidence = 95 / 100

## Confidence interval for the mean, one column at a time

In [ ]:
# The interval is mean ± multiplier × standard error. The standard error, SD / √n, is
# how far the average of a sample this size typically lies from the true average. Read
# the interval as the range of true averages these data are consistent with: it says
# nothing about where single values fall, which the SD describes.
results = {}
for name in variables:
    # Summary data: each column holds the sample size, the mean and the standard
    # deviation, top to bottom.
    size, mean, sd = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:3]
    n = int(round(size))
    se = sd / np.sqrt(n)
    # The t distribution with n - 1 degrees of freedom, because the SD is estimated from
    # the same data: its multiplier is a little above the normal 1.96 at 95%. It assumes
    # independent values from a roughly normal population, or enough of them for the
    # average to be close to normal (the central limit theorem).
    lower, upper = stats.t.interval(confidence, n - 1, loc=mean, scale=se)
    results[name] = {"n": n, "variance": sd**2, "SD": sd, "SE": se,
                     "lower": lower, "mean": mean, "upper": upper}

table = pd.DataFrame(results)
print(table)
print(f"\nConfidence level: {confidence:.1%}")

## What the result says

This run uses only three numbers for each measure: the sample size (50 orders), the mean, and the standard deviation, which is the typical distance of one order from the mean. For delivery time the mean is 24.46 minutes and the standard deviation is 6.49 minutes.

From them comes the standard error, 0.92 minutes: how far a sample mean typically lies from the true mean. The 95% confidence interval, the range of values the data agree with for the true average, runs from 22.6 to 26.3 minutes. The 50 raw orders give exactly the same interval.

So a short summary is enough to build the intervals. But three numbers cannot show an unusual order, so they cannot show whether the interval deserves trust.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.